# Multimodal RAG with ColPali

Skip captioning: embed page images directly with a late-interaction vision retriever.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

**Extra requirements:** `pip install colpali-engine torch pillow` (GPU or Apple Silicon recommended; downloads ~6 GB)

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

ColPali embeds a page *image* into many patch vectors and matches them to query tokens (ColBERT-style late interaction). No OCR or captioning step, so layout, tables and charts are preserved.

In [ ]:
import sys
sys.path.insert(0, "..")
from common import *
import torch
from PIL import Image
from colpali_engine.models import ColPali, ColPaliProcessor

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model_name = "vidore/colpali-v1.3"
model = ColPali.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map=device).eval()
processor = ColPaliProcessor.from_pretrained(model_name)
print("loaded on", device)

In [ ]:
paths = make_figures()
images = [Image.open(p).convert("RGB") for p in paths]

image_embeddings = []
for i in range(0, len(images), 2):                      # small batches to save memory
    batch = processor.process_images(images[i:i + 2]).to(model.device)
    with torch.no_grad():
        image_embeddings.extend(list(torch.unbind(model(**batch).to("cpu"))))
print(len(image_embeddings), "page embeddings")

Score the query against every page with `score_multi_vector` (MaxSim over patches).

In [ ]:
queries = ["Which chunk size gives the best hit rate?", "Which pipeline stage is the slowest?"]
batch_q = processor.process_queries(queries).to(model.device)
with torch.no_grad():
    query_embeddings = list(torch.unbind(model(**batch_q).to("cpu")))

scores = processor.score_multi_vector(query_embeddings, image_embeddings)
for q, row in zip(queries, scores):
    best = int(row.argmax())
    print(f"{q}\n  -> {paths[best].name} (score {float(row[best]):.1f})")

In [ ]:
from IPython.display import display
display(images[int(scores[0].argmax())])

Feed the retrieved page image to a vision LLM (e.g. `llava` through Ollama) to generate the final answer.